In [22]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

from utils import is_casenum

import nltk
from nltk.corpus import stopwords
import re

# Download stopwords if you haven't already
nltk.download('stopwords')

# Load English stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /Users/ekung/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [23]:
meetings_df = pd.read_csv(os.path.join(LOCAL_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [24]:
all_dfs = []

for idx, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]

    # First, get full agenda text file
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda.txt")
    override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-manual-override.txt")
    if (not os.path.exists(input_filepath)) and (not os.path.exists(override_filepath)):
        print("No agenda text file found, skipping.")
        continue
    if os.path.exists(override_filepath):
        with open(override_filepath, 'r', encoding='utf-8') as f:
            full_agenda_text = f.read()
    else:
        with open(input_filepath, 'r', encoding='utf-8') as f:
            full_agenda_text = f.read()

    agenda_text_lines = full_agenda_text.split("\n")

    # Next, get agenda summary JSON
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-summary.json")
    override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-summary-override.json")
    if os.path.exists(override_filepath):
        with open(override_filepath, 'r') as f:
            j = json.load(f)
    elif os.path.exists(input_filepath):
        with open(input_filepath, 'r') as f:
            j = json.load(f)
    else:
        continue
    
    # Make a df
    agenda_items = j["agenda_items"]
    my_df = []
    for item in agenda_items:
        my_row = {
            "year": row["year"],
            "date": date
        }
        for k, v in item.items():
            my_row[k] = v
        my_df.append(my_row)
    my_df = pd.DataFrame(my_df)

    # Find start line within full_agenda_text of each agenda item
    for i, item in my_df.iterrows():
        item_title = item['title'].lower().strip()
        item_no = item['item_no'].lower().replace('.', '').strip()
        pattern = rf"{re.escape(item_no.lower())}[\s.]*{re.escape(item_title.lower())}"
        j = 0
        done = False
        while (not done) and (j < len(agenda_text_lines)):
            line = agenda_text_lines[j].lower()
            if re.search(pattern, line):
                my_df.at[i, 'start_line'] = j
                done = True
            elif re.search(pattern, re.sub(r'\s+', ' ', line).strip()):
                my_df.at[i, 'start_line'] = j
                done = True
            else:
                # check if all non stopwords in title are contained in line
                line_nopunc = re.sub(r'[^\w\s]', '', line)
                title_nopunc = re.sub(r'[^\w\s]', '', item_title.lower())
                line_tokens = [t for t in line_nopunc.split() if t not in stop_words]
                title_tokens = [t for t in title_nopunc.split() if t not in stop_words]
                no_in_line = (len(line_tokens)>0) and (f"{item_no}"==line_tokens[0])
                title_in_line = set(title_tokens).issubset(set(line_tokens))
                if no_in_line and title_in_line:
                    my_df.at[i, 'start_line'] = j
                    done = True
                
                # possible that 4a. is written as 4\n   a. ... check for that
                itemno_alphaonly = re.sub(r'[^a-z]', '', item_no)
                title_tokens_w_alpha = [itemno_alphaonly] + title_tokens
                line_tokens_w_alpha = [itemno_alphaonly] + line_tokens
                line_check_1 = line_tokens[0:len(title_tokens_w_alpha)]
                line_check_2 = line_tokens_w_alpha[0:len(title_tokens_w_alpha)]
                if line_check_1 == title_tokens_w_alpha or line_check_2 == title_tokens_w_alpha:
                    my_df.at[i, 'start_line'] = j
                    done = True

            j += 1
        if not done:
            print(line_tokens)
            print(title_tokens)
            raise ValueError(f"Could not find start line for agenda item {item_no} - {item_title} on {date}")
    
    # Check that start lines are monotonic
    start_line = 0
    for i, item in my_df.iterrows():
        if int(item['start_line']) < start_line:
            raise ValueError(f"Start lines not monotonic for agenda items on {date}")
        start_line = int(item['start_line'])
    
    # Fill in end line of each item
    for i, item in my_df.iterrows():
        if i < len(my_df)-1:
            my_df.at[i, 'end_line'] = my_df.at[i+1, 'start_line'] - 1
        else:
            # Look for indicator of end of agenda with words next meeting of city planning commission...
            j = int(item['start_line'])+1
            done = False
            while (not done) and (j < len(agenda_text_lines)):
                line = agenda_text_lines[j].lower()
                if ('next' in line) and ('meeting' in line) and ('city' in line) and ('planning' in line) and ('commission' in line):
                    my_df.at[i, 'end_line'] = j - 1
                    done = True
                j += 1
            if not done:
                raise ValueError(f"Line containing next city planning commission meeting not found for {date}")
    
    # Extract the full agenda text of the item
    for i, item in my_df.iterrows():
        start_line = int(item['start_line'])
        end_line = int(item['end_line'])
        item_lines = agenda_text_lines[start_line:end_line+1]
        item_text = "\n".join(item_lines).strip()
        my_df.at[i, 'full_agenda_text'] = item_text

    all_dfs.append(my_df)

out_df = pd.concat(all_dfs, ignore_index=True)
out_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc/agenda-summaries.parquet"), index=False)

[]
['citywide', 'design', 'guidelines', 'update', 'presentation']


ValueError: Could not find start line for agenda item 12 - citywide design guidelines update presentation on 2018-07-12

In [ ]:
for i, line in enumerate(agenda_text_lines):
    print(f"{i}: {line}")

0:                                                                                     
1:                                                                                     
2:                                                                                     
3:                               CITY PLANNING COMMISSION                              
4:                            **REVISED REGULAR MEETING AGENDA                         
5:                            THURSDAY, JULY 12, 2018 after 8:30 a.m.                  
6:                        LOS ANGELES CITY COUNCIL CHAMBER, ROOM 340                   
7:                    200 NORTH SPRING STREET, LOS ANGELES, CALIFORNIA 90012           
8:        David H. Ambroz, President                     Vincent P. Bertoni, AICP, Director
9:        Renee Dake Wilson, AIA, Vice President      Kevin J. Keller, AICP, Executive Officer
10:        Caroline Choe, Commissioner                 Lisa M. Webber, AICP, Deputy Director
11:        Vahid